# NER Pipeline Test - First 50 Documents

This notebook runs the complete NER pipeline on the first 50 documents:
1. **batch_ner.py** - Performs Named Entity Recognition
2. **add_semantic_types_from_cui.py** - Adds semantic types to entities
3. **merge_entities_by_umls.py** - Exports entities by UMLS concept
4. **copy_relevant_files.py** - Filters disease-related files

## Directory Structure

```
nlp-histo/                          (project root)
├── langchain-summarization/        (current directory)
│   ├── test_pipeline_50_docs.ipynb (this notebook)
│   └── test_results_50_docs/       (outputs go here)
│       ├── umls_entities/          (all UMLS concepts)
│       └── relevant_texts/         (disease-related only)
├── named-entity-recognition/
├── scripts/
└── database/
```

All outputs are saved to `test_results_50_docs/` within the `langchain-summarization/` folder.

## Setup and Configuration

In [1]:
import sys
import subprocess
import os
from pathlib import Path
from datetime import datetime

# Project root directory (parent of langchain-summarization)
project_root = Path.cwd().parent

# Test output directories (inside langchain-summarization folder)
LANGCHAIN_DIR = Path.cwd()
TEST_OUTPUT_BASE = LANGCHAIN_DIR / "test_results_50_docs"
TEST_UMLS_DIR = TEST_OUTPUT_BASE / "umls_entities"
TEST_RELEVANT_DIR = TEST_OUTPUT_BASE / "relevant_texts"

# Create test directories
TEST_OUTPUT_BASE.mkdir(exist_ok=True)
TEST_UMLS_DIR.mkdir(exist_ok=True)
TEST_RELEVANT_DIR.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Notebook directory: {LANGCHAIN_DIR}")
print(f"Test output directory: {TEST_OUTPUT_BASE}")
print(f"UMLS entities output: {TEST_UMLS_DIR}")
print(f"Relevant texts output: {TEST_RELEVANT_DIR}")
print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Helper Function for Running Scripts

In [2]:
def run_script(script_path, args=None, description=""):
    """
    Run a Python script and capture output.
    
    Args:
        script_path: Path to the script
        args: List of command-line arguments
        description: Description of what the script does
    """
    print("="*80)
    print(f"Running: {description}")
    print(f"Script: {script_path}")
    if args:
        print(f"Args: {' '.join(args)}")
    print("="*80 + "\n")
    
    cmd = [sys.executable, str(script_path)]
    if args:
        cmd.extend(args)
    
    start_time = datetime.now()
    
    try:
        result = subprocess.run(
            cmd,
            cwd=project_root,
            capture_output=True,
            text=True,
            check=True
        )
        
        print(result.stdout)
        if result.stderr:
            print("STDERR:", result.stderr)
        
        elapsed = (datetime.now() - start_time).total_seconds()
        print(f"\n✅ Completed in {elapsed:.1f}s\n")
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Error running script:")
        print(e.stdout)
        print(e.stderr)
        
        elapsed = (datetime.now() - start_time).total_seconds()
        print(f"\nFailed after {elapsed:.1f}s\n")
        return False

print("Helper function loaded ✓")

Helper function loaded ✓


## Step 1: Run Named Entity Recognition (First 50 Documents)

Processes the first 50 documents from the database and performs NER with UMLS mapping.

In [3]:
ner_script = project_root / "named-entity-recognition" / "batch_ner.py"

success = run_script(
    script_path=ner_script,
    args=["--limit", "50", "--min-chars", "50"],
    description="Named Entity Recognition on first 50 documents"
)

if success:
    print("✅ NER completed successfully")
else:
    print("❌ NER failed - check errors above")

Running: Named Entity Recognition on first 50 documents
Script: /Users/emir/Documents/GitHub/nlp-histo/named-entity-recognition/batch_ner.py
Args: --limit 50 --min-chars 50

Loading spaCy model and UMLS Linker into RAM (one-time cost)...
Limiting to first 50 documents
Batch NER Processing Started at: 2026-01-18 13:05:55

Using 4 worker threads (CPU count: 8)
[1/50] [NERWorker_0] Processing PMC7149534...
✓ UMLS Linker initialized successfully.
[2/50] [NERWorker_1] Processing PMC7768511...
✓ UMLS Linker initialized successfully.
[3/50] [NERWorker_2] Processing PMC11674653...[4/50] [NERWorker_3] Processing PMC9355740...
✓ UMLS Linker initialized successfully.

✓ UMLS Linker initialized successfully.
Submitted 50 NER tasks to thread pool

⚠ Document PMC7768511 already has 920 entities in database.
Skipping processing. Use --force to reprocess and replace.

[5/50] [NERWorker_1] Processing PMC6120400...⊘ PMC7768511      - skipped      (  1/ 50 completed)

✓ UMLS Linker initialized successful

## Step 2: Add Semantic Types to Entities

Enriches entities with UMLS semantic type information (e.g., T047: Disease or Syndrome).

In [ ]:
semantic_script = project_root / "scripts" / "add_semantic_types_from_cui.py"

success = run_script(
    script_path=semantic_script,
    args=[],
    description="Add semantic types to entities"
)

if success:
    print("✅ Semantic types added successfully")
else:
    print("❌ Semantic type addition failed - check errors above")

## Step 3: Export Entities by UMLS Concept (First 50 Documents)

Groups all sentences by UMLS concept and exports to JSON and TXT files. Only processes entities from the first 50 documents.

In [4]:
merge_script = project_root / "named-entity-recognition" / "merge_entities_by_umls.py"

success = run_script(
    script_path=merge_script,
    args=["--output-dir", str(TEST_UMLS_DIR), "--limit", "50"],
    description="Merge entities by UMLS CUI (first 50 documents)"
)

if success:
    print("✅ UMLS entity export completed successfully")
    
    # Count output files
    json_files = list(TEST_UMLS_DIR.glob("*.json"))
    txt_files = list(TEST_UMLS_DIR.glob("*.txt"))
    print(f"\n📊 Generated {len(json_files)} JSON files and {len(txt_files)} TXT files")
else:
    print("❌ UMLS entity export failed - check errors above")

Running: Merge entities by UMLS CUI (first 50 documents)
Script: /Users/emir/Documents/GitHub/nlp-histo/named-entity-recognition/merge_entities_by_umls.py
Args: --output-dir /Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs/umls_entities --limit 50

UMLS Entity Merger - Separate JSON Files
Filter: First 50 documents
Minimum occurrences: 1
Output directory: /Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs/umls_entities/

Processing entities from 50 documents

Querying database...
Found 37109 entity occurrences with UMLS mappings

Merging entities by UMLS CUI...
Filtering CUIs with at least 1 occurrences...
Retained 6849 unique UMLS concepts

Writing 6849 JSON and TXT files to /Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs/umls_entities/...
✓ Saved 13698 files to /Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs/umls_entities/

Summary Statistics
Unique UMLS concepts: 6849
Total entity occurrences: 37109

Top 10 Most Frequent UMLS Concepts:
---

## Step 4: Copy Disease-Related Files (From First 50 Documents)

Filters and copies only files related to diseases, neoplasms, and symptoms from the entities extracted from the first 50 documents.

In [5]:
copy_script = project_root / "scripts" / "copy_relevant_files.py"

success = run_script(
    script_path=copy_script,
    args=["--source", str(TEST_UMLS_DIR), "--dest", str(TEST_RELEVANT_DIR)],
    description="Copy disease-related files"
)

if success:
    print("✅ Disease-related files copied successfully")
    
    # Count copied files
    copied_files = list(TEST_RELEVANT_DIR.glob("*.txt"))
    print(f"\n📊 Copied {len(copied_files)} disease-related TXT files")
else:
    print("❌ File copying failed - check errors above")

Running: Copy disease-related files
Script: /Users/emir/Documents/GitHub/nlp-histo/scripts/copy_relevant_files.py
Args: --source /Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs/umls_entities --dest /Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs/relevant_texts

Found 11698 unique disease CUIs in the database.

--- Process Complete ---
Files scanned: 13698
Files copied to '/Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs/relevant_texts': 1591


✅ Completed in 3.0s

✅ Disease-related files copied successfully

📊 Copied 1591 disease-related TXT files


## Pipeline Summary

View statistics and results from the complete pipeline run.

In [6]:
print("="*80)
print("PIPELINE SUMMARY")
print("="*80)
print(f"\nTest output directory: {TEST_OUTPUT_BASE}")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "-"*80)
print("Output Files")
print("-"*80)

# UMLS entities
json_files = list(TEST_UMLS_DIR.glob("*.json"))
txt_files = list(TEST_UMLS_DIR.glob("*.txt"))
print(f"\n📁 {TEST_UMLS_DIR.name}/")
print(f"   • {len(json_files)} JSON files")
print(f"   • {len(txt_files)} TXT files")

# Relevant disease texts
relevant_files = list(TEST_RELEVANT_DIR.glob("*.txt"))
print(f"\n📁 {TEST_RELEVANT_DIR.name}/")
print(f"   • {len(relevant_files)} disease-related TXT files")

# Sample files
if json_files:
    print("\n" + "-"*80)
    print("Sample Files (first 5)")
    print("-"*80)
    for f in sorted(json_files)[:5]:
        size_kb = f.stat().st_size / 1024
        print(f"   • {f.name} ({size_kb:.1f} KB)")
    if len(json_files) > 5:
        print(f"   ... and {len(json_files) - 5} more")

print("\n" + "="*80)
print("✅ Pipeline completed successfully!")
print("="*80)

PIPELINE SUMMARY

Test output directory: /Users/emir/Documents/GitHub/nlp-histo/test_results_50_docs
Completed at: 2026-01-18 13:12:04

--------------------------------------------------------------------------------
Output Files
--------------------------------------------------------------------------------

📁 umls_entities/
   • 6849 JSON files
   • 6849 TXT files

📁 relevant_texts/
   • 1591 disease-related TXT files

--------------------------------------------------------------------------------
Sample Files (first 5)
--------------------------------------------------------------------------------
   • C0000368_3_3_-Diaminobenzidine.json (1.8 KB)
   • C0000723_Abbreviations.json (4.7 KB)
   • C0000726_Abdomen.json (17.6 KB)
   • C0000737_Abdominal_Pain.json (3.3 KB)
   • C0000768_Congenital_Abnormality.json (5.5 KB)
   ... and 6844 more

✅ Pipeline completed successfully!


## Query Database Statistics

Check how many entities were extracted from the first 50 documents.

In [ ]:
sys.path.insert(0, str(project_root))
from database import get_db_connection, Document, Entity, TextElement
from sqlalchemy import func

db = get_db_connection()

with db.session_scope() as session:
    # Get first 50 document IDs
    first_50_docs = session.query(Document.id, Document.pmcid).limit(50).all()
    doc_ids = [doc.id for doc in first_50_docs]
    
    if doc_ids:
        # Count entities for these documents
        entity_count = session.query(func.count(Entity.id)).join(
            TextElement
        ).filter(
            TextElement.document_id.in_(doc_ids)
        ).scalar()
        
        # Count entities with UMLS mapping
        umls_count = session.query(func.count(Entity.id)).join(
            TextElement
        ).filter(
            TextElement.document_id.in_(doc_ids),
            Entity.umls_cui.isnot(None)
        ).scalar()
        
        # Count unique UMLS concepts across all entities
        unique_cuis = session.query(func.count(func.distinct(Entity.umls_cui))).join(
            TextElement
        ).filter(
            TextElement.document_id.in_(doc_ids),
            Entity.umls_cui.isnot(None)
        ).scalar()
    else:
        entity_count = 0
        umls_count = 0
        unique_cuis = 0

print("="*80)
print("DATABASE STATISTICS")
print("="*80)
print(f"\nDocuments processed: {len(first_50_docs)}")
print(f"Total entities extracted: {entity_count:,}")
print(f"Entities with UMLS mapping: {umls_count:,}")
print(f"Unique UMLS concepts: {unique_cuis:,}")
print("\n" + "="*80)

# Show sample PMCIDs
if first_50_docs:
    print("\nFirst 10 PMCIDs processed:")
    for i, doc in enumerate(first_50_docs[:10], 1):
        print(f"  {i:2d}. {doc.pmcid}")
    if len(first_50_docs) > 10:
        print(f"  ... and {len(first_50_docs) - 10} more")

## Explore Sample Output

Read and display a sample UMLS entity file.

In [ ]:
import json

json_files = sorted(TEST_UMLS_DIR.glob("*.json"))

if json_files:
    # Pick a file to explore
    sample_file = json_files[0]
    
    print("="*80)
    print(f"Sample File: {sample_file.name}")
    print("="*80 + "\n")
    
    with open(sample_file, 'r') as f:
        data = json.load(f)
    
    print(f"UMLS CUI: {data['umls_cui']}")
    print(f"Canonical Name: {data['canonical_name']}")
    print(f"Entity Label: {data['entity_label']}")
    print(f"Total Occurrences: {data['total_occurrences']}")
    print(f"Unique Sentences: {len(data['sentences'])}")
    print(f"Text Variants: {', '.join(data['unique_entity_texts'])}")
    
    print("\n" + "-"*80)
    print("First 3 Sentences:")
    print("-"*80 + "\n")
    
    for i, sent in enumerate(data['sentences'][:3], 1):
        print(f"{i}. [{sent['pmcid']}] {sent['section']}")
        print(f"   {sent['sentence'][:200]}..." if len(sent['sentence']) > 200 else f"   {sent['sentence']}")
        print(f"   Entity: '{sent['entity_text']}' (score: {sent['umls_score']:.2f})\n")
else:
    print("No JSON files found to explore.")

## Next Steps

- Review the output files in `test_results_50_docs/`
- Check the database for extracted entities
- If results look good, run on full dataset by removing `--limit 50`
- Use the disease-related files in `relevant_texts/` for downstream analysis